# 04 — Causal ML: Double Machine Learning
## Unbiased Price Elasticity Estimation

**This is the most important notebook in the project.**

---

### The Problem (Economics)
We want to know: *"If I raise the price by 10%, by how much does demand fall?"*

Standard OLS answers a different (biased) question:
*"Among products with different prices, how does demand differ?"*

These differ because **price is endogenous** — it's set based on demand.
Expensive products sell well (Apple iPhone), so OLS sees:
- High price → high demand (correlation)
- But the CAUSAL effect is: high price → lower demand (what we want)

The bias is confounded by quality, brand equity, and pricing strategy.

---

### The Solution (DML)

**Step 1:** Partial out the confounders from demand: `Ỹ = Y - E[Y | controls]`

**Step 2:** Partial out the confounders from price: `T̃ = T - E[T | controls]`

**Step 3:** Regress `Ỹ` on `T̃` → coefficient is the unbiased elasticity

This is the **Frisch-Waugh-Lovell theorem** with ML nuisance models.
The intuition: `T̃` is price variation NOT explained by demand conditions —
it's "as good as random" conditional on the controls.

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SRC_DIR       = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

# Load data and column definitions
train = pd.read_parquet(os.path.join(PROCESSED_DIR, 'train.parquet'))
with open(os.path.join(PROCESSED_DIR, 'col_definitions.json')) as f:
    cols = json.load(f)

Y_col  = cols['Y_col']
T_col  = cols['T_col']
W_cols = cols['W_cols']
X_cols = cols['X_cols']

# Load baseline results for comparison
with open(os.path.join(PROCESSED_DIR, 'baseline_results.json')) as f:
    baseline = json.load(f)

print(f'Loaded train: {train.shape}')
print(f'Baseline OLS elasticity: {baseline["naive_ols_elasticity"]}')

Loaded train: (500000, 20)
Baseline OLS elasticity: 0.004513


In [5]:
from econml.dml import LinearDML
from lightgbm import LGBMRegressor
from sklearn.linear_model import LassoCV
import time
import inspect
import econml

# ─── Prepare matrices ─────────────────────────────────────────────────────────
Y = train[Y_col].values
T = train[T_col].values
W = train[W_cols].fillna(0).values
X = train[X_cols].fillna(0).values

print(f'Y shape: {Y.shape}')
print(f'T shape: {T.shape}')
print(f'W shape: {W.shape}')
print(f'X shape: {X.shape}')

print('\neconml version:', econml.__version__)
print('LinearDML parameters:', list(inspect.signature(LinearDML.__init__).parameters.keys()))

Y shape: (500000,)
T shape: (500000,)
W shape: (500000, 9)
X shape: (500000, 3)

econml version: 0.16.0
LinearDML parameters: ['self', 'model_y', 'model_t', 'featurizer', 'treatment_featurizer', 'fit_cate_intercept', 'linear_first_stages', 'discrete_outcome', 'discrete_treatment', 'categories', 'cv', 'mc_iters', 'mc_agg', 'random_state', 'allow_missing', 'enable_federation', 'use_ray', 'ray_remote_func_options']


In [6]:
# ─── LinearDML — correct API for econml 0.16 ─────────────────────────────────
# In 0.16, the final stage model is set via model_y being a special class
# or by using CausalForestDML. For LinearDML, the final stage is always
# a linear model — we control regularisation via fit_cate_intercept.

dml = LinearDML(
    model_y=model_y,
    model_t=model_t,
    fit_cate_intercept=True,
    discrete_treatment=False,
    cv=3,
    random_state=42,
)

print('DML model configured ✅')
print(f'Nuisance model Y: LightGBM (n_estimators=100)')
print(f'Nuisance model T: LightGBM (n_estimators=100)')
print(f'Final stage:      Linear (built-in to LinearDML)')
print(f'Cross-fitting:    3-fold')
print(f'\nFitting now — 5-15 minutes...')

start = time.time()
dml.fit(Y, T, X=X, W=W)
elapsed = time.time() - start

print(f'\n✅ DML fitted in {elapsed/60:.1f} minutes')

DML model configured ✅
Nuisance model Y: LightGBM (n_estimators=100)
Nuisance model T: LightGBM (n_estimators=100)
Final stage:      Linear (built-in to LinearDML)
Cross-fitting:    3-fold

Fitting now — 5-15 minutes...

✅ DML fitted in 0.7 minutes


In [8]:
# ─── Extract results ──────────────────────────────────────────────────────────
# ATE: Average Treatment Effect = average price elasticity across all observations
ate = dml.ate(X=X)
ate_interval = dml.ate_interval(X=X, alpha=0.05)  # 95% confidence interval

# CATE: Conditional ATE = elasticity for each individual observation
cate = dml.effect(X=X)

print('=' * 55)
print('DOUBLE ML RESULTS — PRICE ELASTICITY')
print('=' * 55)
print(f'\nATE (average elasticity):   {ate:.6f}')
print(f'95% CI lower:               {ate_interval[0]:.6f}')
print(f'95% CI upper:               {ate_interval[1]:.6f}')

print(f'\nCOMPARISON TO BASELINES:')
print(f'  Naive OLS:                {baseline["naive_ols_elasticity"]:.6f}  ← biased')
print(f'  OLS + controls:           {baseline["ols_controls_elasticity"]:.6f}  ← biased')
print(f'  DML (causal):             {ate:.6f}  ← unbiased ✅')
print(f'\n  Endogeneity bias corrected: {abs(ate - baseline["naive_ols_elasticity"]):.6f}')

print(f'\nCATE Summary:')
print(f'  Mean:   {cate.mean():.6f}')
print(f'  Std:    {cate.std():.6f}')
print(f'  Min:    {cate.min():.6f}')
print(f'  Max:    {cate.max():.6f}')
print(f'  % elastic (|ε|>1): {(abs(cate)>1).mean():.1%}')

DOUBLE ML RESULTS — PRICE ELASTICITY

ATE (average elasticity):   -0.082911
95% CI lower:               -0.222609
95% CI upper:               0.056787

COMPARISON TO BASELINES:
  Naive OLS:                0.004513  ← biased
  OLS + controls:           0.005841  ← biased
  DML (causal):             -0.082911  ← unbiased ✅

  Endogeneity bias corrected: 0.087424

CATE Summary:
  Mean:   -0.082911
  Std:    0.251675
  Min:    -1.036697
  Max:    0.464274
  % elastic (|ε|>1): 0.0%


In [ ]:
import joblib

# Save fitted DML model
models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)
joblib.dump(dml, os.path.join(models_dir, 'dml_model.pkl'))

# Save CATE estimates
train['estimated_elasticity'] = cate
train.to_parquet(os.path.join(PROCESSED_DIR, 'train_with_cate.parquet'), index=False)

# Save DML results for notebook 07 (pricing optimizer)
dml_results = {
    'ate': float(ate),
    'ate_ci_lower': float(ate_interval[0]),
    'ate_ci_upper': float(ate_interval[1]),
    'cate_mean': float(cate.mean()),
    'cate_std': float(cate.std()),
    'cate_min': float(cate.min()),
    'cate_max': float(cate.max()),
    'endogeneity_bias': float(abs(ate - baseline['naive_ols_elasticity'])),
}
with open(os.path.join(PROCESSED_DIR, 'dml_results.json'), 'w') as f:
    json.dump(dml_results, f, indent=2)

print('✅ DML model saved       → models/dml_model.pkl')
print('✅ CATE estimates saved  → data/processed/train_with_cate.parquet')
print('✅ DML results saved     → data/processed/dml_results.json')
print('\nNotebook 04 complete — ready for notebook 07 (pricing optimizer)')

✅ DML model saved       → models/dml_model.pkl
✅ CATE estimates saved  → data/processed/train_with_cate.parquet
✅ DML results saved     → data/processed/dml_results.json

Notebook 04 complete — ready for notebook 07 (pricing optimizer)


: 